# PROJET DE TRAITEMENTS DISTRIBUES
Edoardo PICIUCCHI, Aurélien DUVIGNAC-ROSA, Jean-Marc FAUVEL

# Présentation

Le projet consiste en l'étude d'un article intitulé : « CCF : Fast and Scalable Connected Component Computation in MapReduce » (« CCF : Calcul de composantes connexes rapide et scalable en MapReduce »). Cet article a été publié en 2013 par Jimmy Lin et Michael Schatz. Il propose un algorithme de calcul de composantes connexes en utilisant le modèle de programmation MapReduce. Cet algorithme est basé sur l'algorithme de Label Propagation. Il est conçu pour être rapide et scalable, c'est-à-dire qu'il peut être utilisé sur de grands ensembles de données et sur un grand nombre de machines.

L'algorithme a pour objectif d’identifier les sous-graphes connectés au sein d’un graphe G. Le graphe analysé doit être représenté par une collection de paires (N1, N2) où N1 et N2 sont des noeuds du graph G qui ont une connexion entre eux. Le couple (N1, N2) représente une arête du graphe.
L’algorithme présenté dans ce papier va permettre d’identifier des sous-graphes connectés entre eux en modifiant les paires des sous-graphes afin qu’elles soient toutes constituées ainsi : (N, Nmin) où N est un noeud du sous graph et Nmin le noeud de plus petite valeur appartenant au même sous-graphe.
Par ailleurs, l’algorithme ayant pour objet de permettre de traiter des graphes de grande taille, il adopte une programmation distribuée de type MapReduce, permettant ainsi de répartir le traitement sur plusieurs machines et permettre le traitement des données de manière parallèle.

## Objectifs du projet

1. Lire, comprendre et expliquer l’algorithme décrit dans le papier de Hakan Kardes, Siddharth Agrawal, Xin Wang et Ang Sun intitulé « CCF : Fast and Scalable Connected Component Computation in MapReduce » ;
2. Coder l’algorithme en Spark en utilisant à la fois des RDD et des DataFrames ;
3. L’implémentation doit être exécutée en Python, et optionnellement en Scala
4. Effectuer une analyse comparative des versions en RDD et DataFrame sur des graphes de tailles croissantes ;
5. Utilisation de DataBricks pour les petits graphes, et Google Cloud Cluster pour les plus gros (<20 Gb).

La première étape a consisté à implémenter l'algorithme CCF en se limitant exclusivement à l'utilisation de python sans le paradigme spark. Cela nous a permis de mieux appréhender l'algorithme et d'en comprendre les différentes étapes. Cette étape réalisée, nous avons pu utiliser les éléments relatifs à la bibiliothèque `spark`.
L'implémentation décrite ci-dessous permet de réaliser les étapes de l'algorithme CCF décrites dans l'exemple de l'article.
Les résultats obtenus nous ont permis de constater une erreur au niveau de l'exemple. En effet, dans la figure 5.1 de l'article, le lien entre H et G n'est pas transmis dans le graphe présent dans la colonne *reducer*. Pour le prouver il suffit d'ajouter le paramètre `debug` dans le constructeur de l'instance du graph utilisé de sorte à pouvoir afficher les différentes étapes de l'algorithme et notamment les graphes successifs obtenus après chaque itération. Remarquons à cet égard que les graphes successifs sont décrits par l'intermédiaire d'un dictionnaire.
Comme nous pouvons le constater à l'issue de la première itération il manque le lien entre H et G dans l'exemple de l'article.
Cependant le résultat final obtenu est bien identique à celui fourni dans l'article.

In [27]:
class Graph:
  """
  Graphe exemple
  """
  def __init__(self, input_dict=None, debug=False):
    self.graph = input_dict or {}
    self.debug = debug
    self.iterations = 1
    self.ccf()

  def ccf(self):
			"""
			Algorithme CCF
			"""
			previous_graph = None  # Initialisation pour stocker le graphe précédent
			
			while True:
					# Stocker le graphe courant avant la mise à jour
					current_graph = self.graph  
					
					# Effectuer les étapes mapper et reducer
					self.graph = self.mapper()
					self.graph = self.reducer()
					if self.debug:
						print(f"itération {self.iterations} : {self.graph}")
					
					# Comparer le graphe précédent et le graphe actuel
					if previous_graph == self.graph:
							# Si les graphes sont identiques, arrêter la boucle
							if self.debug:
								print(f"Convergence atteinte après {self.iterations} itérations.")
							break
					
					# Mettre à jour le graphe précédent
					previous_graph = self.graph
					self.iterations += 1

  def mapper(self):
			# convert to bidirectional
			# Créer un nouveau dictionnaire pour stocker le graphe bidirectionnel
			bidirectional_graph = {}
			
			# Parcourir les nœuds du graphe original
			for node, neighbors in self.graph.items():
					# Ajouter chaque voisin à la liste des voisins du nœud courant
					if node not in bidirectional_graph:
							bidirectional_graph[node] = []
					for neighbor in neighbors:
							if neighbor not in bidirectional_graph:
									bidirectional_graph[neighbor] = []
							# Ajouter les voisins dans les deux sens
							if neighbor not in bidirectional_graph[node]:
									bidirectional_graph[node].append(neighbor)
							if node not in bidirectional_graph[neighbor]:
									bidirectional_graph[neighbor].append(node)
			
			return bidirectional_graph
  
  def map(self, key, value):
    """
		Fonction map
		"""
    return f"""	emit({key},{value})
								emit({value},{key})
						"""

  def reduce(self, key, values):
			# Initialiser la liste des valeurs et le compteur
			valueList = []
			CounterNewPair = 0
			min = key
			
			# Dictionnaire pour stocker les émissions
			emissions = {}
			
			# Trouver la valeur minimale et remplir valueList
			for value in values:
					if value < min:
							min = value
					valueList.append(value)
			
			# Vérifier si une émission est nécessaire
			if min < key:
					# Ajouter l'émission (key, min) au dictionnaire
					if key not in emissions:
							emissions[key] = []
					emissions[key].append(min)
					
					if self.debug:
					  print(f"emit({key},{min})")
					
					# Ajouter les émissions pour les autres valeurs dans valueList
					for value in valueList:
							if min != value:
									CounterNewPair += 1
									if self.debug:
										print(f"emit({value},{min})")
									
									if value not in emissions:
											emissions[value] = []
									emissions[value].append(min)
			
			# Retourner ou enregistrer les émissions pour usage ultérieur
			return emissions

  def reducer(self):
    emissions = []
    for key, value in self.graph.items():
      emissions.append(self.reduce(key, value))
    return self.merge_dictionaries(emissions)

  def merge_dictionaries(self, dictionaries):
			# Dictionnaire final pour stocker les résultats
			final_dict = {}
			
			# Parcourir les dictionnaires successifs
			for current_dict in dictionaries:
					for key, values in current_dict.items():
							# Si la clé n'existe pas, initialiser une liste vide
							if key not in final_dict:
									final_dict[key] = []
							# Ajouter les valeurs, en évitant les doublons
							for value in values:
									if value not in final_dict[key]:
											final_dict[key].append(value)
			
			return final_dict
    

# Exemple de graphe
input_dict = {
	"A" : ["B"],
	"B" : ["C", "D"],
	"D" : ["E"],
	"F" : ["G"],
	"G" : ["H"],
}

g = Graph(input_dict, debug=True)
print(f"le graphe obtenu est le suivant : {g.graph} après {g.iterations} itérations.")

emit(B,A)
emit(C,A)
emit(D,A)
emit(C,B)
emit(D,B)
emit(E,B)
emit(E,D)
emit(G,F)
emit(H,F)
emit(H,G)
itération 1 : {'B': ['A'], 'C': ['A', 'B'], 'D': ['A', 'B'], 'E': ['B', 'D'], 'G': ['F'], 'H': ['F', 'G']}
emit(B,A)
emit(C,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(B,A)
emit(D,A)
emit(B,A)
emit(E,A)
emit(E,B)
emit(D,B)
emit(G,F)
emit(H,F)
emit(H,F)
emit(G,F)
itération 2 : {'B': ['A'], 'C': ['A'], 'D': ['A', 'B'], 'E': ['A', 'B'], 'G': ['F'], 'H': ['F']}
emit(B,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(D,A)
emit(B,A)
emit(E,A)
emit(B,A)
emit(G,F)
emit(H,F)
itération 3 : {'B': ['A'], 'D': ['A'], 'E': ['A'], 'C': ['A'], 'G': ['F'], 'H': ['F']}
emit(B,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(G,F)
emit(H,F)
itération 4 : {'B': ['A'], 'D': ['A'], 'E': ['A'], 'C': ['A'], 'G': ['F'], 'H': ['F']}
Convergence atteinte après 4 itérations.
le graphe obtenu est le suivant : {'B': ['A'], 'D': ['A'], 'E': ['A'], 'C': ['A'], 'G': ['F'], 'H': ['F']} après 4 itérations.
